# Previsione dei Sinistri Mensili Auto con PROC FORECAST

## Sintesi Esecutiva

Un team di riservazione attuariale ha bisogno di una vista a 12 mesi del numero mensile di sinistri auto che mostri una crescita costante del portafoglio più una marcata impennata invernale. Questo notebook genera cinque anni di conteggi sintetici di sinistri mensili (60 mesi, Gen 2020 - Dic 2024, che vanno da circa 1.460 a 2.780 sinistri), quindi usa **PROC FORECAST** per adattare una baseline autoregressiva stepwise e un modello stagionale Holt-Winters moltiplicativo. Ogni modello produce 12 previsioni puntuali mensili con limiti di confidenza al 95% per la pianificazione della capacità e delle riserve. Il modello stagionale Holt-Winters proietta la domanda più forte uno-due mesi in anticipo (circa 2.780-2.900 sinistri) attenuandosi verso un minimo intorno al passo nove (circa 2.200), mentre la baseline autoregressiva proietta un decadimento più uniforme; entrambe le bande di confidenza si allargano con l'orizzonte.

## Fonti Dati

| Dataset | Righe | Grana | Variabili Chiave | Descrizione |
|---------|------|-------|---------------|-------------|
| `claims` | 60 | Una riga per mese di calendario (Gen 2020 - Dic 2024) | `date` (data SAS, `MONYY7.`), `claim_count` (numerico) | Conteggi sintetici di sinistri auto mensili costruiti da una tendenza di crescita lineare (~9 sinistri/mese), un ciclo annuale sinusoidale, un'impennata invernale additiva a Dic/Gen/Feb, e rumore Gaussiano (`rand('normal')`). Il seed `20240531` lo rende riproducibile. |

# Previsione dei Sinistri Mensili Auto

I team di riservazione e pianificazione della capacità di un assicuratore personal-lines hanno bisogno di una vista in avanti su quanti **sinistri auto** verranno denunciati ogni mese. Il volume dei sinistri in questo libro cresce costantemente man mano che il portafoglio si espande, e ha un picco ogni inverno quando ghiaccio, neve e minore luce diurna aumentano i sinistri da collisione e da vetri.

Questo notebook percorre un flusso di lavoro completo con `PROC FORECAST`:

1. Genera una serie di conteggi di sinistri mensili realistica e completamente sintetica.
2. Adatta una baseline **autoregressiva stepwise (STEPAR)** che cattura tendenza più autocorrelazione.
3. Adatta un modello **Holt-Winters moltiplicativo (WINTERS)** che modella esplicitamente il ciclo stagionale di 12 mesi.
4. Confronta i due modelli e interpreta la previsione in avanti e la banda di confidenza.

Tutto viene eseguito su dati sintetici inline — nessun file esterno o accesso di rete.

## Passo 1 — Genera la serie sintetica dei sinistri

Il DATA step sottostante costruisce **60 osservazioni mensili** (da Gen 2020 a Dic 2024). Per ogni mese combiniamo quattro componenti che rispecchiano un vero libro auto:

- **Tendenza** — una base di ~1.800 sinistri che cresce di ~9 al mese man mano che l'esposizione aumenta.
- **Ciclo annuale** — un termine seno/coseno che dà un'onda stagionale uniforme.
- **Impennata invernale** — un aumento additivo a dicembre, gennaio e febbraio.
- **Rumore** — `rand('normal', 0, 90)` per una variabilità mese su mese realistica.

`call streaminit()` fissa il flusso casuale così il notebook è riproducibile. La variabile `date` è una vera data SAS costruita con `INTNX` e formattata `MONYY7.`, e l'istruzione `ID date INTERVAL=MONTH` la nomina come identificatore temporale. Le prime 14 righe stampate sotto si collocano approssimativamente tra 1.460 e 2.450 sinistri.

In [1]:
DATI claims;
    CHIAMARE streaminit(20240531);
    FARE i = 0 FINO_A 59;
        /* Costruisce una vera data SAS mensile così INTERVAL=MONTH si allinea */
        date = intnx('month', '01JAN2020'd, i);
        FORMATO date monyy7.;

        month_idx = mod(i, 12) + 1;          /* 1 = Gen ... 12 = Dic */

        trend  = 1800 + 9 * i;               /* base di esposizione crescente */
        season = 260 * sin((month_idx - 1) / 12 * 2 * constant('pi'))
               + 150 * cos((month_idx - 1) / 12 * 2 * constant('pi'));
        winter = 220 * (month_idx IN (12, 1, 2));   /* impennata ghiaccio/neve */
        noise  = rand('normal', 0, 90);

        claim_count = round(trend + season + winter + noise);
        SE_COND claim_count < 0 ALLORA claim_count = 0;
        USCITA;
    FINE;
    MANTENERE date claim_count;
ESEGUIRE;

PROCEDURA STAMPARE DATI=claims(obs=14) ETICHETTA;
    ETICHETTA date = "Mese" claim_count = "Sinistri Denunciati";
    TITOLO "Primi 14 Mesi di Sinistri Auto Sintetici";
ESEGUIRE;

                                        Primi 14 Mesi di Sinistri Auto Sintetici                                        

  Obs   Mese  Sinistri Denunciati
    1  21915                 2178
    2  21946                 2281
    3  21975                 2252
    4  22006                 1974
    5  22036                 2067
    6  22067                 1805
    7  22097                 1697
    8  22128                 1619
    9  22159                 1462
   10  22189                 1688
   11  22220                 1713
   12  22250                 2008
   13  22281                 2116
   14  22312                 2451

... 46 more observations (showing 14 of 60)




NOTE: DATA claims


NOTE: Wrote claims (60 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=claims

NOTE: PROC PRINT completed: 14 observations printed, 2 variables


## Passo 2 — Baseline autoregressiva stepwise (METHOD=STEPAR)

`METHOD=STEPAR` è il default. Prima adatta un modello di tendenza temporale — qui `TREND=2` per una tendenza lineare — poi applica un'**autoregressione stepwise** ai residui, inserendo e mantenendo i ritardi AR in base alla significatività. `AR=4` limita l'ordine autoregressivo candidato a quattro ritardi, il che è più che sufficiente per una serie mensile con momentum di breve termine.

Opzioni chiave usate qui:

- `LEAD=12` — prevede 12 mesi oltre i dati.
- `ALPHA=0.05` — limiti di confidenza al 95%.
- `OUTFULL` — impila le righe storiche `ACTUAL` insieme alle righe dell'orizzonte di previsione nel dataset `OUT=` (piuttosto che le sole righe di previsione).
- `OUTLIMIT` — aggiunge le **colonne** di limite di confidenza inferiore/superiore (`L95_claim_count`, `U95_claim_count`).
- `ID date INTERVAL=MONTH` — nomina l'identificatore temporale e dichiara che la serie è mensile.

In [2]:
PROCEDURA forecast DATI=claims
        out=fc_stepar
        METHOD=stepar trend=2 ar=4
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    VARIABILE claim_count;
ESEGUIRE;

PROCEDURA STAMPARE DATI=fc_stepar(obs=10) ETICHETTA;
    ETICHETTA date            = "Mese"
          claim_count     = "Sinistri Denunciati"
          l95_claim_count = "Limite Inferiore 95%"
          u95_claim_count = "Limite Superiore 95%";
    TITOLO "Output STEPAR: Righe Effettive, Stimate e di Previsione";
ESEGUIRE;

                                        Primi 14 Mesi di Sinistri Auto Sintetici                                        

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                                Output STEPAR: Righe Effettive, Stimate e di Previsione                                 

  Obs   Mese  _TYPE_  Sinistri Denunciati  Limite Inferiore 95%  Limite Superiore 95%
    1  21915  ACTUAL          2121.816667                     .                     .
    2  21946  ACTUAL          2167.711419                     .                     .
    3  21975  ACTUAL          2254.781177                     .                     .
    4  22006  ACTUAL          2228.505912                     .                     .
    5  22036  ACTUAL          1978.152296                     .                     .
    6  22067  ACTUAL          2030.606563                     .                     .
    7  22097  ACTUAL          1864.520418    


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: PROC PRINT data=fc_stepar

NOTE: PROC PRINT completed: 10 observations printed, 5 variables


### Lettura del dataset OUT=

Il dataset `OUT=` impila le righe tramite una colonna `_TYPE_` e porta i limiti di confidenza come **colonne** laterali:

| Elemento | Tipo | Significato |
|---------|------|-------------|
| `_TYPE_ = 'ACTUAL'` | riga | Il `claim_count` osservato per ciascuno dei 60 mesi storici. |
| `_TYPE_ = 'FORECAST'` | riga | Le 12 previsioni puntuali per l'orizzonte di previsione. |
| `L95_claim_count` / `U95_claim_count` | colonna | Limiti di confidenza inferiore / superiore al 95%, popolati sulle righe `FORECAST` (mancanti sulle righe `ACTUAL`). Il livello numerico riflette `ALPHA=`. |

Quindi l'`OUT=` stampato qui contiene 72 righe: 60 righe storiche `ACTUAL` seguite da 12 righe di orizzonte `FORECAST`. Le prime dieci righe mostrate sopra sono tutte `ACTUAL`, con le colonne di limite di confidenza mancanti perché i limiti si applicano solo alle righe di previsione.

> **Nota sul motore.** SAS `OUTFULL` scrive anche una riga `FORECAST` within-sample a un passo in avanti per ogni mese storico, e `OUTRESID` aggiunge righe `_TYPE_='RESIDUAL'`. La PROC FORECAST di Jenner non emette ancora queste righe di stima/residuo in-sample (tracciato come gap test `400979`), quindi questo notebook legge solo lo storico `ACTUAL` e l'orizzonte `FORECAST` in avanti.

## Passo 3 — Modello stagionale Holt-Winters (METHOD=WINTERS)

Il modello STEPAR cattura tendenza e autocorrelazione di breve termine ma non modella esplicitamente il rialzo ricorrente di dicembre-febbraio. Per una serie con un ritmo annuale chiaro, l'**Holt-Winters moltiplicativo** è lo strumento migliore.

`METHOD=WINTERS` decompone la serie in livello, tendenza lineare (`TREND=2`), e un **fattore stagionale moltiplicativo**. `SEASONS=12` dichiara un ciclo stagionale di 12 periodi (mensile). Richiediamo di nuovo un `LEAD` di 12 mesi con limiti al 95% e `OUTFULL OUTLIMIT` in modo che l'output sia allineato con l'esecuzione STEPAR.

Poiché il motore avanza l'`ID` di previsione di un'unità per passo (piuttosto che rispettare `INTERVAL=MONTH` per le date dell'orizzonte — gap test `400979`), la cella sottostante esamina la previsione per **mesi in anticipo (passo 1-12)** invece di affidarsi alle date di calendario stampate.

In [3]:
PROCEDURA forecast DATI=claims
        out=fc_winters
        METHOD=winters seasons=12 trend=2
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    VARIABILE claim_count;
ESEGUIRE;

/* Mantiene l'orizzonte di previsione a 12 mesi e lo indicizza per passo (1..12). */
DATI horizon;
    IMPOSTARE fc_winters;
    CONSERVARE months_ahead 0;
    SE_COND _type_ = 'FORECAST';
    months_ahead + 1;
    MANTENERE months_ahead claim_count l95_claim_count u95_claim_count;
ESEGUIRE;

PROCEDURA STAMPARE DATI=horizon ETICHETTA;
    ETICHETTA months_ahead     = "Mesi in Anticipo"
          claim_count       = "Sinistri Previsti"
          l95_claim_count   = "Limite Inferiore 95%"
          u95_claim_count   = "Limite Superiore 95%";
    TITOLO "Previsione Holt-Winters a 12 Mesi (per passo)";
ESEGUIRE;

                                Output STEPAR: Righe Effettive, Stimate e di Previsione                                 

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                                     Previsione Holt-Winters a 12 Mesi (per passo)                                      

  Obs  Mesi in Anticipo  Sinistri Previsti  Limite Inferiore 95%  Limite Superiore 95%
    1                 1         2783.07951           2577.844742           2988.314278
    2                 2        2897.355557           2607.109764           3187.601349
    3                 3        2805.272075           2449.795029            3160.74912
    4                 4        2664.498049           2254.028514           3074.967585
    5                 5        2628.810136           2169.891244           3087.729029
    6                 6         2548.39319           2045.672732           3051.113649
    7                 7        2391.64


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: DATA horizon


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote horizon (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=horizon

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Passo 4 — Confronta i due modelli fianco a fianco

Il modo più chiaro per giudicare se il modello stagionale merita il suo posto è mettere la sua previsione a 12 mesi accanto alla baseline STEPAR, passo per passo. Estraiamo le righe `FORECAST` da entrambi i dataset `OUT=`, indicizziamo ciascuna per mesi-in-anticipo, e le uniamo così che la divergenza sia visibile a colpo d'occhio.

I due metodi concordano sul livello generale ma discordano sulla **forma**: Holt-Winters porta il ritmo annuale nell'orizzonte (un livello più alto nell'orizzonte iniziale che si attenua verso un minimo a metà orizzonte e risale di nuovo), mentre STEPAR — che modella la stagionalità solo indirettamente tramite ritardi AR — decade più uniformemente verso la media della serie. Il divario tra loro ad ogni passo è il segnale stagionale che STEPAR lascia sul tavolo.

> SAS normalmente guiderebbe questo controllo di adeguatezza con i residui a un passo `OUTRESID` (`_TYPE_='RESIDUAL'`). Jenner non popola ancora quelle righe (gap test `400979`), quindi confrontiamo direttamente le due previsioni in avanti — una diagnostica che usa solo l'output che il motore produce effettivamente.

In [4]:
/* Orizzonte STEPAR, indicizzato per mesi-in-anticipo. */
DATI stepar_h;
    IMPOSTARE fc_stepar;
    CONSERVARE months_ahead 0;
    SE_COND _type_ = 'FORECAST';
    months_ahead + 1;
    stepar = claim_count;
    MANTENERE months_ahead stepar;
ESEGUIRE;

/* Orizzonte WINTERS, indicizzato per mesi-in-anticipo. */
DATI winters_h;
    IMPOSTARE fc_winters;
    CONSERVARE months_ahead 0;
    SE_COND _type_ = 'FORECAST';
    months_ahead + 1;
    winters = claim_count;
    MANTENERE months_ahead winters;
ESEGUIRE;

/* Unisce i due e mostra il gap stagionale che STEPAR non coglie. */
DATI COMPARE;
    UNIRE stepar_h winters_h;
    PER months_ahead;
    seasonal_gap = winters - stepar;
ESEGUIRE;

PROCEDURA STAMPARE DATI=COMPARE ETICHETTA;
    ETICHETTA months_ahead = "Mesi in Anticipo"
          stepar        = "Previsione STEPAR"
          winters       = "Previsione Winters"
          seasonal_gap  = "Winters - STEPAR";
    FORMATO stepar winters seasonal_gap 8.0;
    TITOLO "STEPAR vs Holt-Winters: Confronto Previsione a 12 Mesi";
ESEGUIRE;

                                 STEPAR vs Holt-Winters: Confronto Previsione a 12 Mesi                                 

  Obs  Mesi in Anticipo  Previsione STEPAR  Previsione Winters  Winters - STEPAR
    1                 1               2619                2783               164
    2                 2               2537                2897               361
    3                 3               2384                2805               421
    4                 4               2214                2664               450
    5                 5               2089                2629               540
    6                 6               2010                2548               539
    7                 7               1982                2392               410
    8                 8               1995                2240               245
    9                 9               2031                2197               166
   10                10               2075                2354      


NOTE: DATA stepar_h


NOTE: Read 72 rows from fc_stepar.
NOTE: Wrote stepar_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA winters_h


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote winters_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA compare

NOTE: Stream 1 processed 12 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 12 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote compare (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=compare

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Passo 5 — Prevedi ogni linea di business in una volta (elaborazione BY)

Le esecuzioni di riservazione reali coprono diversi prodotti. Con i dati ordinati per `line_of_business`, un'istruzione `BY` fa sì che `PROC FORECAST` adatti un **modello stagionale indipendente per ogni gruppo** in un'unica chiamata. Qui sintetizziamo un libro Auto (volume base più alto) e un libro Casa (base più bassa) e prevediamo entrambi sei mesi in avanti. `OUTALL` scrive l'intero set di righe — lo storico `ACTUAL` e l'orizzonte `FORECAST` — insieme alle colonne di limite per ogni gruppo, e teniamo solo le righe `FORECAST` per la tabella sottostante.

In [5]:
DATI multi_book;
    CHIAMARE streaminit(20240531);
    LUNGHEZZA line_of_business $6;
    FARE lob = 1 FINO_A 2;
        SE_COND lob = 1 ALLORA line_of_business = 'Auto';
        ALTRIMENTI            line_of_business = 'Casa';
        FARE i = 0 FINO_A 47;
            date = intnx('month', '01JAN2021'd, i);
            FORMATO date monyy7.;
            mi   = mod(i, 12) + 1;
            BASE = (lob = 1) * 2000 + (lob = 2) * 1400;
            claim_count = round(BASE + 8 * i
                + 200 * sin((mi - 1) / 12 * 2 * constant('pi'))
                + 180 * (mi IN (12, 1, 2))
                + rand('normal', 0, 70));
            USCITA;
        FINE;
    FINE;
    MANTENERE line_of_business date claim_count;
ESEGUIRE;

PROCEDURA ORDINARE DATI=multi_book;
    PER line_of_business date;
ESEGUIRE;

PROCEDURA forecast DATI=multi_book
        out=fc_by
        METHOD=winters seasons=12 trend=2
        LEAD=6 ALPHA=0.05
        outall;
    PER line_of_business;
    id date interval=month;
    VARIABILE claim_count;
ESEGUIRE;

PROCEDURA STAMPARE DATI=fc_by(obs=12) ETICHETTA;
    ETICHETTA line_of_business = "Linea di Business"
          date              = "Mese"
          claim_count       = "Sinistri";
    DOVE _type_ = 'FORECAST';
    TITOLO "Previsioni a 6 Mesi per Linea di Business";
ESEGUIRE;

                                 STEPAR vs Holt-Winters: Confronto Previsione a 12 Mesi                                 


BY Group: line_of_business=Auto

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6


BY Group: line_of_business=Casa

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6



                                       Previsioni a 6 Mesi per Linea di Business                                        

  Obs  Linea di Business   Mese    _TYPE_     Sinistri  L95_CLAIM_COUNT  U95_CLAIM_COUNT  RESIDUAL_CLAIM_COUNT
    1  Auto               23742  FORECAST  2524.596717      2359.095846      2690.097589                     .
    2  Auto               23773  FORECAST  2604.418724      2370.365147        2838.4723                     .
    3  Auto               23801  FORECAST  2576.092313      2289.436395       2862.74823                     .
    4  Auto               2383


NOTE: DATA multi_book


NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC SORT data=multi_book

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 96 rows from multi_book.
NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC FORECAST data=multi_book

NOTE: BY Group: line_of_business=Auto
NOTE: Using Python for FORECAST estimation
NOTE: BY Group: line_of_business=Casa
NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 108 observations.
NOTE: PROC PRINT data=fc_by

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 6 observations printed, 7 variables


## Interpretare i risultati

**Cosa dicono i modelli al team di riservazione:**

- **STEPAR** riproduce la deriva verso l'alto e il momentum di breve termine, ma la sua previsione a 12 mesi decade uniformemente da circa 2.620 (passo 1) verso circa 1.980 a metà orizzonte prima di risalire verso circa 2.140 — non riproduce un picco invernale netto, perché tratta la stagionalità solo tramite ritardi autoregressivi. È una baseline rapida ragionevole.
- **WINTERS** con `SEASONS=12` porta il ritmo annuale direttamente attraverso i suoi fattori stagionali moltiplicativi: la sua previsione è più alta nell'orizzonte iniziale (circa 2.780 al passo 1, circa 2.900 al passo 2), si attenua verso un minimo vicino al passo 9 (circa 2.200), e risale di nuovo entro il passo 12 (circa 2.800). Rispetto alla baseline STEPAR si colloca **150-660 sinistri più in alto** per la maggior parte dell'orizzonte (vedi la colonna `Winters - STEPAR` nel Passo 4) — quel divario è il segnale stagionale che il modello autoregressivo tralascia.
- La **banda di confidenza al 95%** (colonne `L95_*`/`U95_*`, controllate da `ALPHA=`) si allarga man mano che l'orizzonte si estende — per il modello WINTERS da una larghezza di circa 410 sinistri al passo 1 a circa 1.420 al passo 12 — il segnale onesto che le stime a orizzonte più lontano portano più incertezza di quelle a breve termine. I riservisti dovrebbero accantonare capitale rispetto al limite superiore, non solo alla previsione puntuale.
- L'**elaborazione BY** produce le previsioni Auto e Casa in un unico passaggio, ognuna con il proprio adattamento stagionale. Il libro Auto prevede nell'intervallo approssimativo di 2.510-2.600 mentre il libro Casa si colloca vicino a 1.870-2.030, così ogni linea mantiene il proprio livello e la propria forma stagionale — lo schema che il team estenderebbe a ogni prodotto nel portafoglio.

**Conclusione:** per una serie di conteggi di sinistri con un chiaro ciclo annuale, `METHOD=WINTERS SEASONS=12` è la scelta idiomatica; la baseline STEPAR è un utile controllo di buon senso, e `OUTFULL`/`OUTLIMIT` più un confronto di modelli passo-per-passo permettono all'attuario di dimensionare la riserva in avanti con la sua banda di incertezza.

> **Nota di fedeltà del motore.** Questo notebook documenta due limitazioni attuali di Jenner PROC FORECAST (gap test `400979`): l'`ID` dell'orizzonte di previsione viene avanzato di un'unità per passo piuttosto che per `INTERVAL=MONTH`, quindi le date di previsione stampate non sono i mesi di calendario 2025 previsti (esaminiamo l'orizzonte per passo invece); e `OUTRESID`/`OUTALL` non popolano ancora le righe a un passo in avanti `_TYPE_='RESIDUAL'`, quindi la diagnostica dei residui è sostituita da un confronto diretto tra i due modelli. Le previsioni puntuali e i limiti di confidenza stessi sono prodotti e sono ciò su cui si basa la narrazione sopra.